# Spark: конфигурация приложения


Показать, где задаются настройки Spark, как посмотреть фактические значения и чем параметры по умолчанию отличаются от явных переопределений.

Нужен `SparkSessionContext`; примеры используют только встроенные настройки Spark и не изменяют таблицы.

Ограничения: Когда нужно объяснить поведение Spark, воспроизвести учебный запуск или аккуратно настроить чтение и перемешивание. Значения по умолчанию зависят от версии Spark и режима запуска; не копируйте настройки производственного кластера вслепую и не задавайте число частей только по размеру кластера.


## Основные понятия
Параметры Spark зависят от версии, режима запуска и сборки. Поэтому сначала читайте фактическое значение в текущем приложении, а затем задавайте только те настройки, которые нужны конкретной задаче.


## Примеры


**1. Параметры по умолчанию**

Прочитайте настройки вычислительных ресурсов и SQL. Второй аргумент `get` используется только как справочное значение, если настройка не задана явно.

`spark.driver.memory` и `spark.executor.memory` задают объём памяти управляющего процесса и каждого рабочего процесса; значения по умолчанию зависят от режима запуска, поэтому в примере приведены типичные справочные значения. `memoryOverhead` добавляет память вне JVM-кучи для Python и нативных библиотек. `spark.executor.cores` задаёт число ядер на рабочий процесс, а `spark.cores.max` ограничивает суммарное число ядер в автономном кластере. `spark.sql.shuffle.partitions` обычно имеет значение 200, адаптивное выполнение включено в современных сборках Spark, а размер файловой части чтения обычно ограничен 128 МиБ. Всегда ориентируйтесь на `actual`: Docker, JupyterHub и проект могут переопределить встроенное значение.


In [ ]:
from lakehouse_spark import SparkSessionContext

parameters = {
    "spark.driver.memory": "1g",
    "spark.driver.memoryOverhead": "384m",
    "spark.executor.memory": "1g",
    "spark.executor.memoryOverhead": "384m",
    "spark.executor.cores": "1",
    "spark.cores.max": "infinity",
    "spark.sql.shuffle.partitions": "200",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.files.maxPartitionBytes": "134217728",
    "spark.sql.autoBroadcastJoinThreshold": "10485760",
}

with SparkSessionContext("example-spark-default-configuration") as spark:
    for key, documented_default in parameters.items():
        actual = spark.conf.get(key, "<не задано>")
        print({"parameter": key, "actual": actual, "typical_default": documented_default})


**2. Память и процессоры**

Задайте ресурсы для отдельного приложения и выведите фактический профиль управляющего и рабочих процессов. Это демонстрация параметров, а не гарантия доступного ресурса кластера.

Управляющий процесс координирует приложение и выполняет код Python, поэтому ему нужна отдельная память. Рабочие процессы выполняют распределённые задачи; их память и число ядер задаются на один процесс, а `spark.cores.max` ограничивает суммарное использование ядер. В JupyterHub проекта по умолчанию применяются 512 МиБ для управляющего процесса, 1 ГиБ для рабочего процесса, одно ядро рабочего процесса и максимум одно ядро приложения. Для Airflow действуют отдельные настройки, не совпадающие с JupyterHub. Параметры памяти управляющего процесса нужно задавать до создания SparkContext; в уже запущенной тетрадке они могут не измениться.


In [ ]:
from lakehouse_spark import SparkSessionContext

configuration = {
    "spark.driver.memory": "1g",
    "spark.driver.memoryOverhead": "512m",
    "spark.executor.memory": "2g",
    "spark.executor.memoryOverhead": "512m",
    "spark.executor.cores": "2",
    "spark.cores.max": "4",
}

with SparkSessionContext("example-spark-resources", config=configuration) as spark:
    for key in configuration:
        print(key, "=", spark.conf.get(key))


**3. Явные настройки приложения**

Передайте настройки в контекст до первого действия Spark и проверьте, что они применились.

Словарь `config` передаётся при создании сеанса. Число 8 выбрано только для небольшой учебной задачи: слишком малое число создаёт крупные части и слабый параллелизм, а слишком большое увеличивает служебные расходы. Настройки памяти управляющего процесса, как правило, должны быть заданы до запуска приложения; настройки рабочих процессов зависят от управляющего узла и могут быть ограничены инфраструктурой.


In [ ]:
from lakehouse_spark import SparkSessionContext

configuration = {
    "spark.sql.shuffle.partitions": "8",
    "spark.sql.adaptive.enabled": "true",
    "spark.executor.cores": "2",
    "spark.cores.max": "4",
}

with SparkSessionContext("example-spark-explicit-configuration", config=configuration) as spark:
    for key in configuration:
        print(key, "=", spark.conf.get(key))


**4. Размер партиций при чтении**

Покажите влияние ограничения размера входного файла на количество частей чтения. Значение задаётся в байтах.

Типичное значение `spark.sql.files.maxPartitionBytes` — 128 МиБ. В примере установлено 64 МиБ: крупные файлы могут быть разделены на большее число частей чтения. Параметр не объединяет физические файлы Iceberg и не гарантирует точный размер каждой части.


In [ ]:
from lakehouse_spark import SparkSessionContext

configuration = {"spark.sql.files.maxPartitionBytes": str(64 * 1024 * 1024)}
with SparkSessionContext("example-spark-file-partitions", config=configuration) as spark:
    print("maxPartitionBytes:", spark.conf.get("spark.sql.files.maxPartitionBytes"))
    print("Значение в мегабайтах:", int(spark.conf.get("spark.sql.files.maxPartitionBytes")) / 1024 / 1024)


**5. Перемешивание и адаптивное выполнение**

Сопоставьте число частей после перемешивания и включение адаптивного выполнения. Малое значение удобно для учебной задачи, но не является универсальным для кластера.

Адаптивное выполнение может объединять слишком мелкие части после перемешивания на основании фактического объёма данных. Для `spark.sql.adaptive.enabled` и `spark.sql.adaptive.coalescePartitions.enabled` типичное значение — `true`. Начальное число частей всё равно задаётся через `spark.sql.shuffle.partitions`.


In [ ]:
from lakehouse_spark import SparkSessionContext

configuration = {
    "spark.sql.shuffle.partitions": "8",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
}

with SparkSessionContext("example-spark-shuffle-configuration", config=configuration) as spark:
    for key in configuration:
        print(key, "=", spark.conf.get(key))


**6. Безопасный выбор настроек**

Соберите небольшой профиль настроек для чтения Parquet и покажите его в журнале без секретов.

Векторизованное чтение Parquet обычно включено и быстрее построчного чтения. Отключать его следует только для диагностики несовместимого файла или конкретной ошибки чтения. Профиль настроек нужно сохранять вместе с результатами опыта, чтобы запуск можно было воспроизвести.


In [ ]:
from lakehouse_spark import SparkSessionContext

configuration = {
    "spark.sql.parquet.enableVectorizedReader": "false",
    "spark.sql.files.maxPartitionBytes": str(128 * 1024 * 1024),
}

with SparkSessionContext("example-spark-safe-profile", config=configuration) as spark:
    profile = {key: spark.conf.get(key) for key in configuration}
    print(profile)
